# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrew417/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row = one content item, for one client, on one calendar day (grain: report_date + client_hash_id + content_hash_id). This slice is month=2026-03, spanning 2026-03-01 to 2026-03-31, 9,841,378 rows.


In [ ]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {  
    
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':        f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':         f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':     f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

REL_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
con.sql(f"""
    SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {REL_MARCH}
""").show()

con.sql(f"SELECT * FROM {REL_MARCH} LIMIT 5").show()

┌─────────┬────────────┬────────────┐
│    n    │   min_d    │   max_d    │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context (IDs/grouping only):** report_date, client_hash_id, content_hash_id, month, client_has_ga4, ga4_data_available, client_has_gsc, gsc_data_available

**Feature (knowable before decision point):** gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_pageviews, ga4_engaged_sessions, scroll_events, sessions_ai — all aggregated over the prior/prev-30 window only

**Label/proxy (never a feature):** gsc_impressions aggregated over the last-30 window — used to compute is_declining = (imp_last30 < 0.8 * imp_prev30)

**Excluded:** rows where client_has_ga4 = false — GA4 columns are structurally NULL (no tracking access), not a real zero; verified: 3,018,741 of 9,841,378 March rows (30.7%)


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT
        client_has_ga4,
        ga4_data_available,
        COUNT(*) AS n,
        AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS pct_sessions_null
    FROM {REL_MARCH}
    GROUP BY 1, 2
""").show()

con.sql(f"""
    SELECT ga4_data_available, AVG(ga4_sessions) AS avg_sessions, COUNT(*) AS n
    FROM {REL_MARCH}
    WHERE client_has_ga4 = true
    GROUP BY 1
""").show()

┌────────────────┬────────────────────┬─────────┬───────────────────┐
│ client_has_ga4 │ ga4_data_available │    n    │ pct_sessions_null │
│    boolean     │      boolean       │  int64  │      double       │
├────────────────┼────────────────────┼─────────┼───────────────────┤
│ false          │ NULL               │ 3018741 │               1.0 │
│ true           │ false              │ 6408671 │               0.0 │
│ true           │ true               │  413966 │               0.0 │
└────────────────┴────────────────────┴─────────┴───────────────────┘

┌────────────────────┬────────────────────┬─────────┐
│ ga4_data_available │    avg_sessions    │    n    │
│      boolean       │       double       │  int64  │
├────────────────────┼────────────────────┼─────────┤
│ false              │                0.0 │ 6408671 │
│ true               │ 3.1398907156626388 │  413966 │
└────────────────────┴────────────────────┴─────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain check:** GROUP BY (report_date, client_hash_id, content_hash_id) HAVING COUNT(*) > 1 returns 0 rows — confirms one row per content item, per client, per day.

**Counts + date span:** 9,841,378 rows, 2026-03-01 to 2026-03-31 — confirms the full March window with no gaps at the edges.

**Availability (IS TRUE):** 6,822,637 rows (69.3%) have client_has_ga4 = true; 3,018,741 rows (30.7%) have client_has_ga4 = false, where GA4 columns are structurally NULL — confirms the exclusion in step 2.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {REL_MARCH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").show()

con.sql(f"""
    SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {REL_MARCH}
""").show()

con.sql(f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN client_has_ga4 IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
        SUM(CASE WHEN client_has_ga4 IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_unavailable
    FROM {REL_MARCH}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
└─────────────┴────────────────┴─────────────────┴───────┘
                          0 rows                        

┌─────────┬────────────┬────────────┐
│    n    │   min_d    │   max_d    │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┬─────────────────┐
│  total  │ ga4_available │ ga4_unavailable │
│  int64  │    int128     │     int128      │
├─────────┼───────────────┼─────────────────┤
│ 9841378 │       6822637 │         3018741 │
└─────────┴───────────────┴─────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This March 2026 slice comes from an unbalanced panel — clients started tracking at different times, so any single month mixes clients with over a year of history alongside newly onboarded ones. 30.7% of rows (3,018,741 of 9,841,378) have client_has_ga4 = false, meaning GA4 columns are structurally NULL, not zero — this isn't random missingness, it's tied to which clients never had GA4 access set up. At this share, GA4 features are still usable for the ~6.8M rows that have them, but any model or score relying on GA4 signals needs a separate path (flag, segment, or GSC-only fallback) for the excluded third — treating GA4 as a universal feature would silently misrepresent 30% of the data as "zero engagement" instead of "unmeasured." A single month also can't reveal seasonality or safely validate a 30-day trend label without confirming the surrounding days are covered by this same table.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.